# 🚀 JE AI Audio Studio — Colab API Server

This notebook runs the existing `api/server.py` from `main` on Google Colab and exposes it through a temporary HTTPS Cloudflare Tunnel for the Expo Android client.

**Phone:** Expo app only → **Colab:** FastAPI + AI/audio processing.


In [ ]:
import os, sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
print('CUDA:', end=' ')
try:
    import torch
    print(torch.cuda.is_available(), torch.__version__)
except Exception as e:
    print('not available yet:', e)


In [ ]:
repo = '/content/JE-AI-Audio-Studio'
if not os.path.exists(repo):
    !git clone https://github.com/jejamalodasi/JE-AI-Audio-Studio.git $repo
else:
    %cd $repo
    !git fetch origin main
    !git reset --hard origin/main
%cd $repo
!git rev-parse --short HEAD


In [ ]:
# Install the project's base + optional AI layer, plus the API runtime.
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt
!python -m pip install -q -r requirements-ai.txt
!python -m pip install -q 'fastapi>=0.115,<1' 'uvicorn>=0.30,<1' 'python-multipart>=0.0.9,<1'
print('✅ Dependencies installed')


In [ ]:
# Import/syntax smoke test. No model weights are downloaded here.
import py_compile
py_compile.compile('api/server.py', doraise=True)
from api.server import app
print('✅ FastAPI app:', app.title)
print('✅ Routes:', [r.path for r in app.routes])


In [ ]:
# Start FastAPI in the background. Keep the notebook cell alive only long enough to launch it.
import subprocess, time, os, signal
log_path = '/content/je_ai_api.log'
proc = subprocess.Popen([
    sys.executable, '-m', 'uvicorn', 'api.server:app',
    '--host', '0.0.0.0', '--port', '8000'
], stdout=open(log_path, 'w'), stderr=subprocess.STDOUT)
time.sleep(4)
print('API PID:', proc.pid)
print(open(log_path).read()[-4000:])


In [ ]:
# Local health check from inside Colab.
import urllib.request, json
data = urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=10).read().decode()
print('✅ /health:', data)


In [ ]:
# Install Cloudflare Tunnel (temporary public HTTPS URL; no account/token required for a quick tunnel).
!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared
!/tmp/cloudflared --version


In [ ]:
# Start the temporary HTTPS tunnel and extract its public URL.
import subprocess, re, time
tunnel_log = '/content/cloudflared.log'
tunnel = subprocess.Popen(['/tmp/cloudflared','tunnel','--url','http://127.0.0.1:8000'], stdout=open(tunnel_log,'w'), stderr=subprocess.STDOUT)
public_url = None
for _ in range(30):
    time.sleep(1)
    txt = open(tunnel_log, errors='ignore').read()
    m = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', txt)
    if m:
        public_url = m.group(0)
        break
print('Tunnel PID:', tunnel.pid)
print('EXPO_PUBLIC_API_URL =', public_url or 'NOT FOUND — inspect /content/cloudflared.log')
if public_url:
    print('Health URL:', public_url + '/health')


## 📱 Android / Expo

Copy the printed `EXPO_PUBLIC_API_URL` value into the mobile project's environment/configuration. **Do not add `/api`**; the app should use the base URL, for example `https://xxxx.trycloudflare.com`.

Keep this Colab runtime running while the Android app is using the API. A Colab restart or runtime shutdown will change the temporary tunnel URL.


In [ ]:
# Final public health check.
if public_url:
    import urllib.request
    print(urllib.request.urlopen(public_url + '/health', timeout=20).read().decode())
else:
    print('No public URL was detected. Check /content/cloudflared.log')


In [ ]:
# Diagnostics if anything fails.
print('--- API LOG ---')
print(open('/content/je_ai_api.log', errors='ignore').read()[-8000:])
print('--- CLOUDFLARED LOG ---')
print(open('/content/cloudflared.log', errors='ignore').read()[-8000:])
